# 02 — Kiểm chứng full PDP baseline trên CUDA

**Trạng thái: đã bắt đầu; run `e1f12f37` FAIL ở environment/pip check (Ninja), full CUDA chưa chạy.** Tạo ngày 09/10/2026. Tiếp nối [notebook 01](01_processor_loader_pdp_acceptance.ipynb)
đã đạt `PASS_LIMITED_PDP_FOUNDATION` trên máy RTX 3060 (smoke của notebook 01 chạy CPU).

Mục tiêu: full pretrained **ResNet50, encoder 6/decoder 6, 300 queries, d_model 256, 4 feature levels**;
shared/private prompt 100×10, DDL 0.15, query loss 0.1; freeze backbone/encoder/decoder theo baseline hiện có.
Không bật LoRA/K=3/FSA/freeze shared bổ sung. Kiểm tra trên ảnh thật train Task 1 của release đã khóa.

**Không bắt buộc RTX 4090 cho notebook này.** Dùng CUDA trên RTX 3060 trước, batch vật lý 1, FP32, TF32 tắt.
VRAM thực tế được ghi lại; không giả định mọi bản 3060 đều đủ cho 800. Mỗi resolution chạy trong subprocess riêng.
OOM → `PARTIAL`, giữ kết quả 640; chuyển gate còn thiếu sang GPU đủ bộ nhớ hoặc RTX 4090 Vast.ai.
Không tự giảm resolution/prompt/layers hoặc bật AMP để biến OOM thành PASS.

Đạt cả 640/800 chỉ cho `PASS_FULL_PDP_CUDA_SMOKE`: kiểm chứng full baseline Task 1, không chứng nhận convergence,
5-task teacher/PPG, ConvNeXt, toàn bộ đường CLI/runner hay môi trường Vast.ai chưa chạy thực tế.
Native custom kernel có trường nghiệm thu riêng; CUDA PyTorch fallback vẫn có thể đạt smoke.


## 1. Chuẩn bị weights và chọn kernel

Chọn Python 3.10 của Conda `pdp`, có [dependencies](../../docs/environment/README.md).
Notebook tự chuẩn bị checkpoint public [SenseTime/deformable-detr](https://huggingface.co/SenseTime/deformable-detr/tree/main)
qua [script tải](../../tools/download_pdp_pretrained.py). **Không cần tải dataset COCO hoặc chép đường dẫn thủ công.**
Mặc định để `CONFIG['pretrained_dir']` rỗng: gate pretrained sẽ tải `config.json`, `preprocessor_config.json`,
`model.safetensors` (weights khoảng 161 MB), tự lấy đường dẫn cache và khóa commit SHA cho run.
Cache ở `runs/pretrained/hf-cache/`, không đưa vào git; lần sau tái sử dụng files đã tải.
Không cài package và không cần đăng nhập/key cho checkpoint public này.

Nếu muốn chuẩn bị trước từ repo root, chạy:

```bash
conda activate pdp
python -m tools.download_pdp_pretrained
```

Không cần copy đường dẫn in ra: notebook dùng cùng cache. `pretrained_offline=True` dùng cache và không gọi mạng;
`pretrained_revision` có thể là full commit hash để lặp lại run. Local snapshot do bạn chuẩn bị vẫn dùng được
qua `AUTOCHECKOUT_PRETRAINED_DIR`/`pretrained_dir`; khi có đường dẫn đó, notebook không gọi downloader.
Theo [HF Hub 0.20.3](https://huggingface.co/docs/huggingface_hub/v0.20.3/guides/download), snapshot revision và
SHA256 của từng file được lưu vào `pretrained_download.json` và artifacts; không sửa files trong cache.

Custom kernel tùy chọn: mặc định `require_custom_kernel=False`, ghi nhận kernel/fallback thật.
Đặt True nếu muốn gate bắt buộc kernel native; cần toolkit/compiler/ninja tương thích như README môi trường.
Import model có thể build extension lần đầu; build output ở run directory, không phải bước cài dependencies.

Nếu gate environment báo `ninja 1.11.1.1 is not supported on this platform`, làm theo
[phần sửa Ninja](../../docs/environment/README.md#cấu-hình-project-và-lỗi-ninja-ở-notebook-02):

```bash
conda activate pdp
python -m pip install --no-cache-dir --force-reinstall --only-binary=:all: ninja==1.13.0
python -m pip check
```

Requirements đã đổi Ninja sang 1.13.0; model stack giữ nguyên. Restart kernel sau sửa và chạy setup với run ID mới.
Giữ outputs của run FAIL cũ hoặc lưu snapshot trước rerun; không bỏ gate pip check.


In [1]:
import os, sys, json, uuid, subprocess, hashlib, shutil, time, platform
from pathlib import Path
from importlib import metadata

REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists()
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from autocheckout.io import load_json, save_json, md5_file
from autocheckout.model_acceptance import RunEvidence, now
from autocheckout.taskcfg import TaskConfig
from tools.download_pdp_pretrained import DEFAULT_REVISION, sha256_file

RAW = Path(os.getenv("AUTOCHECKOUT_DATA_ROOT", str(REPO / "data/archive"))).resolve()
RELEASE = Path(
    os.getenv(
        "AUTOCHECKOUT_RELEASE_ROOT",
        str(REPO / "data/processed/rpc_100-4x25_seed0_native_v1"),
    )
).resolve()
CONFIG = {
    "seed": 0,
    "resolutions": [640, 800],
    "batch_size": 1,
    "sample_count": 2,
    "precision": "32-true",
    "tf32": False,
    "workers": 0,
    "require_custom_kernel": False,
    "pretrained_dir": os.getenv(
        "AUTOCHECKOUT_PRETRAINED_DIR", ""
    ),  # optional override; empty automatically downloads the public checkpoint
    "pretrained_revision": DEFAULT_REVISION,  # resolved commit SHA is recorded after download
    "pretrained_offline": False,  # True: reuse cache only
    "foundation_summary": os.getenv("AUTOCHECKOUT_FOUNDATION_SUMMARY", ""),
    "raw_root": str(RAW),
    "release_root": str(RELEASE),
}
assert CONFIG["batch_size"] >= 1 and CONFIG["sample_count"] >= 2 * CONFIG["batch_size"]
assert CONFIG["resolutions"] and set(CONFIG["resolutions"]) <= {640, 800}
ARTIFACT = REPO / "runs/modeling/full_pdp_baseline" / str(uuid.uuid4())
REQUIRED = [
    "environment",
    "foundation",
    "release_lock",
    "pretrained_files",
    "cuda_640",
    "cuda_800",
    "sources_preserved",
]
RUN = RunEvidence(ARTIFACT, CONFIG, REQUIRED)
RUN.data.update(
    purpose="Full pretrained PDP Task 1 CUDA smoke and checkpoint/resume",
    python=sys.version,
    executable=sys.executable,
    host=platform.node(),
    git_commit=subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
    ).strip(),
    git_dirty=bool(
        subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=REPO, text=True
        ).strip()
    ),
    full_pdp_training_accepted=False,
    convnext_accepted=False,
    vast_gpu_accepted=False,
)
RUN.persist()
shutil.copy2(
    REPO / "notebooks/modeling/02_full_pdp_baseline_acceptance.ipynb",
    ARTIFACT / "notebook_source.ipynb",
)
for name in [
    "full_baseline_acceptance.py",
    "engine.py",
    "main.py",
    "checkpointing.py",
    "inference.py",
]:
    shutil.copy2(REPO / "pdp" / name, ARTIFACT / name)
print("Run:", ARTIFACT.name, "Started:", RUN.data["started_at"], "Artifacts:", ARTIFACT)
shutil.copy2(REPO / "tools/download_pdp_pretrained.py", ARTIFACT / "download_pdp_pretrained.py")


Run: ed32adc8-9d66-403f-9016-919f3c8f9363 Started: 2026-10-09T16:17:51+07:00 Artifacts: /home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363


PosixPath('/home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363/download_pdp_pretrained.py')

## 2. Environment và foundation trước đó

Kiểm tra pins và `pip check` trong đúng interpreter; không gọi GPU trong kernel notebook để tránh giữ CUDA context.
Foundation summary phải có đủ gates PASS. Có thể chỉ định `foundation_summary` khi chuyển máy;
checksum release vẫn kiểm tra lại độc lập. Không đổi run cũ RUNNING/INTERRUPTED thành PASS.


In [2]:
with RUN.gate("environment") as evidence:
    assert sys.version_info[:2] == (3, 10), "Chọn kernel Python 3.10 của pdp"
    pins = {
        "torch": "2.2.2",
        "torchvision": "0.17.2",
        "transformers": "4.37.2",
        "timm": "0.9.12",
        "pytorch-lightning": "2.1.3",
        "packaging": "24.2",
        "typing-extensions": "4.13.2",
    }
    versions = {k: metadata.version(k) for k in pins}
    assert all(versions[k].split("+")[0] == v for k, v in pins.items()), versions
    check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], cwd=REPO, text=True, capture_output=True
    )
    (ARTIFACT / "pip_check.log").write_text(check.stdout + check.stderr)
    assert check.returncode == 0, check.stdout + check.stderr
    evidence.update(packages=versions, pip_check_returncode=check.returncode)

with RUN.gate("foundation") as evidence:
    if CONFIG["foundation_summary"]:
        foundation_path = Path(CONFIG["foundation_summary"]).resolve()
    else:
        root = REPO / "runs/modeling/processor_loader_pdp"
        marker = load_json(root / "latest.json")
        foundation_path = root / marker["run_id"] / "summary.json"
    foundation = load_json(foundation_path)
    required_foundation = [
        "environment",
        "release_lock",
        "task_gt_separation",
        "processor_640",
        "processor_800",
        "loader_640",
        "loader_800",
        "pdp_smoke",
        "baseline_regression",
        "lightning_resume",
        "sources_preserved",
    ]
    assert foundation["status"] == "PASS_LIMITED_PDP_FOUNDATION"
    assert all(
        foundation["gates"].get(k, {}).get("status") == "PASS"
        for k in required_foundation
    )
    assert foundation["gates"]["release_lock"]["protocol"] == [100, 25, 25, 25, 25]
    shutil.copy2(foundation_path, ARTIFACT / "foundation_summary.json")
    evidence.update(
        run_id=foundation["run_id"],
        status_of_source=foundation["status"],
        summary_sha256=hashlib.sha256(foundation_path.read_bytes()).hexdigest(),
    )

## 3. Release/checksums và batch train Task 1

Kiểm tra nguồn, task config, split/test lock và toàn bộ task JSON. Chỉ chọn ảnh trong `train_task_1.json`,
labels 0–99; không đọc full GT làm batch và không chạy test holdout. Hash các ảnh đã chọn và source code.
Hai ảnh mặc định → hai batch/epoch, hai epoch, một manual step: tổng 5 optimizer steps cho mỗi resolution.


In [3]:
with RUN.gate("release_lock") as evidence:
    LOCK = load_json(REPO / "configs/data/rpc_100-4x25_seed0_native_v1.json")
    TASKS = RELEASE / "tasks/real"
    task_config_path = RELEASE / "task_config.json"
    task_manifest_path = TASKS / "manifest.json"
    assert (
        hashlib.sha256(task_manifest_path.read_bytes()).hexdigest()
        == LOCK["task_manifest_sha256"]
    )
    assert (
        hashlib.sha256(task_config_path.read_bytes()).hexdigest()
        == LOCK["input"]["task_config_sha256"]
    )
    TASK_CONFIG = TaskConfig.load(task_config_path)
    assert [t.num_classes for t in TASK_CONFIG.data_tasks] == [100, 25, 25, 25, 25]
    assert (TASK_CONFIG.num_slots, TASK_CONFIG.num_classes) == (224, 225)
    assert TASK_CONFIG.seed == 0 and LOCK["image_root"] == "data/archive"
    assert not LOCK["synthetic_included"] and LOCK["coordinates"] == "native_pixels"
    SOURCE_BEFORE = {}
    for file_name, digest in LOCK["input"]["source_sha256"].items():
        path = RAW / file_name
        assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, path
        SOURCE_BEFORE[str(path)] = ("sha256", digest)
    for file_name, digest in LOCK["split_md5"].items():
        path = RELEASE / "splits" / file_name
        assert md5_file(path) == digest, path
        SOURCE_BEFORE[str(path)] = ("md5", digest)
    TASK_MANIFEST = load_json(task_manifest_path)
    for file_name, details in TASK_MANIFEST["files"].items():
        path = TASKS / file_name
        assert md5_file(path) == details["md5"], path
        SOURCE_BEFORE[str(path)] = ("md5", details["md5"])
    assert md5_file(TASKS / "test_full.json") == LOCK["test_full_md5"]
    SOURCE_BEFORE[str(task_config_path)] = (
        "sha256",
        LOCK["input"]["task_config_sha256"],
    )
    SOURCE_BEFORE[str(task_manifest_path)] = ("sha256", LOCK["task_manifest_sha256"])
    evidence.update(
        protocol=[100, 25, 25, 25, 25],
        slots=224,
        classifier_outputs=225,
        task_config=str(task_config_path),
        checked_files=len(SOURCE_BEFORE),
    )
    canonical = load_json(TASKS / "train_task_1.json")
    assert all(0 <= a["category_id"] < 100 for a in canonical["annotations"])
    CONFIG["image_ids"] = [
        i["id"]
        for i in sorted(canonical["images"], key=lambda x: x["id"])[
            : CONFIG["sample_count"]
        ]
    ]
    selected = {
        i["id"]: i for i in canonical["images"] if i["id"] in CONFIG["image_ids"]
    }
    for row in selected.values():
        path = (RAW / row["file_name"]).resolve()
        assert path.is_relative_to(RAW) and path.is_file()
        assert (
            row["width"] == row["height"]
        ), "Square RPC inputs expected in this batch smoke"
        SOURCE_BEFORE[str(path)] = (
            "sha256",
            hashlib.sha256(path.read_bytes()).hexdigest(),
        )
    evidence.update(
        real_train_image_ids=CONFIG["image_ids"], current_task_labels=[0, 99]
    )

for folder in ["pdp", "autocheckout"]:
    for path in (REPO / folder).rglob("*.py"):
        SOURCE_BEFORE[str(path)] = (
            "sha256",
            hashlib.sha256(path.read_bytes()).hexdigest(),
        )
SOURCE_BEFORE[str(REPO / "configs/data/rpc_100-4x25_seed0_native_v1.json")] = (
    "sha256",
    hashlib.sha256(
        (REPO / "configs/data/rpc_100-4x25_seed0_native_v1.json").read_bytes()
    ).hexdigest(),
)
save_json(ARTIFACT / "sources_before.json", SOURCE_BEFORE, indent=2)

## 4. Local pretrained snapshot và provenance

Chưa có snapshot → gọi script tải public, tự nhận đường dẫn; lỗi tải/cache → FAIL kèm log. Worker sẽ lấy full architecture từ config,
không tải ResNet qua timm lần nữa vì detector snapshot đã chứa backbone.
Loading report phải chứng minh core backbone/encoder/decoder/projection/bbox đã nạp;
chỉ classifier khác số lớp và các tham số PDP mới được phép khởi tạo. Không coi `ignore_mismatched_sizes` là đủ.


In [4]:
with RUN.gate("pretrained_files") as evidence:
    if not CONFIG["pretrained_dir"]:
        print("Preparing public SenseTime/deformable-detr checkpoint (cache reused when available)...", flush=True)
        manifest = ARTIFACT / "pretrained_download.json"
        command = [sys.executable, "-m", "tools.download_pdp_pretrained",
                   "--cache-dir", str(REPO / "runs/pretrained/hf-cache"),
                   "--manifest", str(manifest), "--revision", CONFIG["pretrained_revision"]]
        if CONFIG["pretrained_offline"]:
            command.append("--offline")
        download = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
        (ARTIFACT / "pretrained_download.log").write_text(download.stdout + download.stderr)
        assert download.returncode == 0, "Checkpoint download failed; see pretrained_download.log: " + download.stderr[-3000:]
        receipt = load_json(manifest)
        CONFIG["pretrained_dir"] = receipt["snapshot_dir"]
        CONFIG["pretrained_revision"] = receipt["resolved_revision"]
        evidence.update(repo_id=receipt["repo_id"], resolved_revision=receipt["resolved_revision"],
                        source_url=receipt["source_url"], download_manifest=str(manifest))
        print("Checkpoint ready:", CONFIG["pretrained_dir"])
    weights = Path(CONFIG["pretrained_dir"]).expanduser().resolve()
    assert weights.is_dir(), weights
    required_files = [weights / "config.json", weights / "preprocessor_config.json"]
    candidates = [
        p
        for p in [weights / "model.safetensors", weights / "pytorch_model.bin"]
        if p.is_file()
    ]
    assert (
        candidates
    ), "Need model.safetensors or pytorch_model.bin (unsharded detector snapshot)"

    SOURCE_BEFORE[str(REPO / "tools/download_pdp_pretrained.py")] = (
        "sha256", sha256_file(REPO / "tools/download_pdp_pretrained.py")
    )
    hashes = {}
    for path in required_files + candidates:
        assert path.is_file(), path
        digest = sha256_file(path)
        hashes[path.name] = digest
        SOURCE_BEFORE[str(path)] = ("sha256", digest)
    CONFIG["pretrained_dir"] = str(weights)
    evidence.update(directory=str(weights), sha256=hashes)
    save_json(ARTIFACT / "pretrained_sha256.json", hashes, indent=2)
    save_json(ARTIFACT / "config.json", CONFIG, indent=2)
    save_json(ARTIFACT / "sources_before.json", SOURCE_BEFORE, indent=2)

Preparing public SenseTime/deformable-detr checkpoint (cache reused when available)...
Checkpoint ready: /home/sakana/Code/AutoCheckout-CL/runs/pretrained/hf-cache/models--SenseTime--deformable-detr/snapshots/83ecd26945199939cb82806f988debdb71e6f43e


## 5. Full CUDA gates 640 và 800

Worker dùng chính `engine.local_trainer.common_step`, optimizer, `main.make_train_loader`,
`inference.predict_batch` và `ResumeCheckpoint`; full pretrained factory được cấu hình/nạp local có report.
Không dùng tiny fixture trong run này. Processor được truyền chung cho loader/trainer ở từng resolution.
Đường CLI/runner mặc định chưa được notebook này thay đổi hay nghiệm thu.

Mỗi gate kiểm tra: CUDA FP32; losses CE/bbox/GIoU/DDL/query finite; gradient classifier/query/shared/private;
optimizer không thiếu/trùng params; cập nhật classifier; frozen hashes; two-pass inference;
ngắt sau step đầu rồi khôi phục weights/optimizer/scheduler/prototype state/global step qua Lightning;
prediction trước/sau reload tương đương. State đang giữ trong RAM bị cố ý làm sai trước restore để chứng minh reload.
Ghi peak allocated/reserved VRAM, GPU/runtime, timing, config, load report và logs.

Nếu gate FAIL/OOM vẫn có `result.json`; đọc log cuối. Khi kernel bị hard-kill, không có PASS được suy diễn.
Nếu notebook bị ngắt, subprocess được terminate và gate INTERRUPTED; có thể chạy cell tổng hợp cuối để chốt report.


In [5]:
for resolution in CONFIG["resolutions"]:
    name = f"cuda_{resolution}"
    output = ARTIFACT / str(resolution)
    output.mkdir(exist_ok=True)
    command = [
        sys.executable,
        str(REPO / "pdp/full_baseline_acceptance.py"),
        "--config",
        str(ARTIFACT / "config.json"),
        "--resolution",
        str(resolution),
        "--output",
        str(output),
    ]
    RUN.record(name, {"status": "RUNNING", "started_at": now(), "command": command})
    with (output / "worker.log").open("w") as log:
        process = subprocess.Popen(
            command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                print(f'{name}: đang chạy; log: {output/"worker.log"}', flush=True)
                time.sleep(15)
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            RUN.record(
                name,
                {
                    "status": "INTERRUPTED",
                    "finished_at": now(),
                    "reason": "Notebook interrupted worker",
                },
            )
            RUN.finish()
            raise
    result_path = output / "result.json"
    result = (
        load_json(result_path)
        if result_path.exists()
        else {"status": "FAIL", "error": "Worker exited without result"}
    )
    if result.get("status") not in {"PASS", "OOM", "FAIL", "INTERRUPTED"}:
        result.update(status="FAIL", error="Worker ended before completing the gate")
    if process.returncode != 0 and result.get("status") == "PASS":
        result.update(status="FAIL", error="Nonzero worker exit despite PASS result")
    result.update(
        returncode=process.returncode, finished_at=result.get("finished_at") or now()
    )
    RUN.record(name, result)
    print(
        name,
        result["status"],
        "peak allocated GiB:",
        round(result.get("peak_allocated_bytes", 0) / (1024**3), 3),
    )
    if result["status"] != "PASS":
        print(
            "\n".join(
                (output / "worker.log").read_text(errors="replace").splitlines()[-35:]
            )
        )

cuda_640: đang chạy; log: /home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363/640/worker.log
cuda_640: đang chạy; log: /home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363/640/worker.log
cuda_640 PASS peak allocated GiB: 3.487
cuda_800: đang chạy; log: /home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363/800/worker.log
cuda_800: đang chạy; log: /home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363/800/worker.log
cuda_800: đang chạy; log: /home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363/800/worker.log
cuda_800 PASS peak allocated GiB: 4.224


## 6. Source preservation, tổng hợp và nghiệm thu

Hash lại ảnh, sources/release/holdout/weights/code đã đọc. Gate OOM hoặc không chạy 800 → PARTIAL;
lỗi nội dung → FAIL. Không chỉnh kết quả cũ. `custom_kernel_accepted` chỉ True khi cả hai resolution PASS
và thực sự dùng kernel native không fallback. Prototype counts có thể bằng 0 sau smoke ngắn;
khôi phục prototype state không chứng nhận đã học prototype đủ 100 SKU.

Khi đạt: tiếp tục notebook adapter ConvNeXt-V2-Base, rồi pilot PDP nền/ConvNeXt trên 4090 Vast.ai cùng split/budget.
Khi PARTIAL vì OOM: giữ report này, chạy lại với cùng weights/protocol trên GPU đủ VRAM; dùng run ID mới.
Luôn lưu notebook có outputs sau thực chạy; bản `notebook_source.ipynb` trong run là snapshot trước execution.


In [6]:
with RUN.gate("sources_preserved") as evidence:
    for filename, (algorithm, expected) in SOURCE_BEFORE.items():
        path = Path(filename)
        actual = md5_file(path) if algorithm == "md5" else sha256_file(path)
        assert actual == expected, f"Source changed: {path}"
    evidence.update(
        checked_files=len(SOURCE_BEFORE), sources_and_holdout_unchanged=True
    )

both = all(
    RUN.data["gates"].get(f"cuda_{r}", {}).get("status") == "PASS" for r in [640, 800]
)
RUN.data.update(
    full_model_640_800_forward_backward_accepted=both,
    custom_kernel_accepted=both
    and all(
        RUN.data["gates"][f"cuda_{r}"].get("custom_kernel_accepted", False)
        for r in [640, 800]
    ),
    limitations=[
        "Task 1 short smoke, not convergence or full five-task teacher/PPG acceptance.",
        "Uses real modules with notebook-selected processor; default training CLI/runner still pending.",
        "No ConvNeXt, synthetic augmentation, calibration/evaluation or Vast.ai acceptance.",
        "Resume restores state and step counts; exact shuffled sample order is not restored.",
    ],
)
summary = RUN.finish()
# Overall acceptance also requires environment/provenance/source preservation gates.
RUN.data["full_model_640_800_forward_backward_accepted"] = (
    both and summary["status"] == "PASS_FULL_PDP_CUDA_SMOKE"
)
RUN.data["custom_kernel_accepted"] = (
    RUN.data["custom_kernel_accepted"]
    and summary["status"] == "PASS_FULL_PDP_CUDA_SMOKE"
)
RUN.persist()
save_json(ARTIFACT / "summary.json", RUN.data, indent=2)
save_json(
    ARTIFACT.parent / "latest.json",
    {
        "run_id": ARTIFACT.name,
        "status": summary["status"],
        "summary": str(ARTIFACT / "summary.json"),
        "finished_at": summary["finished_at"],
    },
    indent=2,
)
rows = [
    {
        "resolution": r,
        "status": RUN.data["gates"].get(f"cuda_{r}", {}).get("status", "NOT_RUN"),
        "peak_allocated_GiB": round(
            RUN.data["gates"].get(f"cuda_{r}", {}).get("peak_allocated_bytes", 0)
            / 1024**3,
            3,
        ),
        "kernel_native": RUN.data["gates"]
        .get(f"cuda_{r}", {})
        .get("custom_kernel_accepted", False),
    }
    for r in [640, 800]
]
print(json.dumps(rows, ensure_ascii=False, indent=2))
print(
    "Status:",
    summary["status"],
    "Finished:",
    summary["finished_at"],
    "Artifacts:",
    ARTIFACT,
)
print(
    "Save this notebook to retain outputs; no full training/ConvNeXt/Vast.ai acceptance is implied."
)

[
  {
    "resolution": 640,
    "status": "PASS",
    "peak_allocated_GiB": 3.487,
    "kernel_native": false
  },
  {
    "resolution": 800,
    "status": "PASS",
    "peak_allocated_GiB": 4.224,
    "kernel_native": false
  }
]
Status: PASS_FULL_PDP_CUDA_SMOKE Finished: 2026-10-09T16:19:09+07:00 Artifacts: /home/sakana/Code/AutoCheckout-CL/runs/modeling/full_pdp_baseline/ed32adc8-9d66-403f-9016-919f3c8f9363
Save this notebook to retain outputs; no full training/ConvNeXt/Vast.ai acceptance is implied.
